<a href="https://colab.research.google.com/github/KanthiPhoosorn/CareMind/blob/feature%2Fpatient-staff-chatbots/CareMind_Custom_Citation_Based_Medical_Chatbot_for_Patient.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# CareMind — Custom Citation-Based Medical Chatbot for Patient (No External LLMs)

This notebook demonstrates how to build a **fully local, custom medical chatbot system** for CareMind using ONLY:

- Your own hospital datasets
- Retrieval pipelines
- Rule-based reasoning
- Citation generation
- TF-IDF semantic matching
- No OpenAI
- No Gemini
- No Qwen
- No Ollama
- No hosted APIs

---

# Important Clarification

This notebook does **NOT** create a frontier large language model from scratch.

Instead, it builds:

```text
Medical Retrieval + Citation Engine
+ Workflow Intelligence
+ Rule-Based Clinical Responses
+ Search-Based Patient Assistant
```

This is the correct MVP architecture for CareMind.

---

# Why This Approach Matters

Training a true medical LLM from scratch requires:

- billions of tokens
- massive GPU clusters
- months of training
- extremely large budgets

For CareMind MVP, the correct architecture is:

```text
Structured Retrieval
+ Citation Engine
+ Medical Rules
+ Search Intelligence
```

This produces:
- safer outputs
- deterministic citations
- better PHI control
- easier hospital deployment


In [9]:

# Core imports

import os
import re
import glob
import json
import csv
import html
import pandas as pd

from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict, Any
from xml.etree import ElementTree as ET

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity



# Step 1 — Load Hospital Data

This notebook loads the provided `data.zip` structure.

Expected layout:

```text
data/
  AN1/
    DoctorProgress_note.xlsx
    NURSE_Note.xlsx
    order (drug).xlsx
    order (lab).xlsx
    order (xray).xlsx
```


In [1]:
import os
import zipfile

zip_path = '/content/data.zip'
target_path = '/content/caremind_data'

# Create the directory if it exists in the notebook environment
try:
    os.makedirs(target_path, exist_ok=True)
except PermissionError:
    target_path = os.path.join(os.getcwd(), 'caremind_data')
    os.makedirs(target_path, exist_ok=True)

# Unzip the data when running in a notebook environment that provides the archive
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(target_path)
    print(f'Successfully extracted {zip_path} to {target_path}')
else:
    print(f'Warning: {zip_path} not found. Using local workspace data path: {target_path}')


In [2]:
import os
print(f'Current Working Directory: {os.getcwd()}')
print('Files in /content:')
print(os.listdir('/content')) if os.path.exists('/content') else print('/content not found')

Current Working Directory: /workspaces/CareMind
Files in /content:
/content not found


In [6]:
# Check the extracted directory structure
target_path = '/content/caremind_data'
for root, dirs, files in os.walk(target_path):
    level = root.replace(target_path, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 4 * (level + 1)
    for f in files[:3]: # Show first 3 files
        print(f'{subindent}{f}')

In [10]:
# Find all clinical files in the correctly identified path
clinical_patterns = [
    "**/*.xlsx",
    "**/*.xls",
    "**/*.xlsm",
    "**/*.csv",
    "**/*.tsv",
    "**/*.json",
    "**/*.txt",
    "**/*.md",
    "**/*.xml",
    "**/*.html",
    "**/*.htm",
]

clinical_files = []
for pattern in clinical_patterns:
    clinical_files.extend(glob.glob(os.path.join(target_path, pattern), recursive=True))

clinical_files = sorted(set(clinical_files))
excel_files = clinical_files  # Backward-compatible alias for later cells

print(f"Found {len(clinical_files)} clinical data files across multiple formats.")


Found 0 clinical data files across multiple formats.


In [ ]:
clinical_files[:10]


[]

# Step 2 — Convert Clinical Files into Retrieval Chunks

Clinical data may arrive as spreadsheets, CSV/TSV exports, JSON payloads, text notes, Markdown summaries, XML feeds, or HTML pages.

This step normalizes each file into searchable chunks so downstream retrieval works across HIS formats, not just Excel rows.


In [11]:
@dataclass
class ClinicalChunk:
    chunk_id: str
    patient_an: str
    source_file: str
    source_ref: str
    file_format: str
    text: str
    row_index: Any = ""
    sheet_name: str = ""


def clean_text(value: Any) -> str:
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    text = str(value).strip()
    return re.sub(r"\s+", " ", text)


def extract_patient_an(file_path: str) -> str:
    path_text = file_path.replace("\\", "/")
    match = re.search(r"AN\d+", path_text, flags=re.IGNORECASE)
    if match:
        return match.group(0).upper()
    return os.path.basename(os.path.dirname(file_path)) or "UNKNOWN"


def make_chunk_text(pairs: List[str]) -> str:
    text = " | ".join(pair for pair in pairs if clean_text(pair))
    return re.sub(r"\s+", " ", text).strip()


def chunk_text_blocks(text: str, min_length: int = 8) -> List[str]:
    blocks = []
    for block in re.split(r"(?:\n\s*\n|\r\n\s*\r\n)", text):
        block = re.sub(r"\s+", " ", block).strip()
        if len(block) >= min_length:
            blocks.append(block)
    return blocks


def flatten_json(value: Any, prefix: str = "") -> List[str]:
    rows = []
    if isinstance(value, dict):
        for key, item in value.items():
            nested_prefix = f"{prefix}{key}" if prefix else str(key)
            rows.extend(flatten_json(item, nested_prefix + "."))
    elif isinstance(value, list):
        for index, item in enumerate(value, start=1):
            nested_prefix = f"{prefix}{index}"
            rows.extend(flatten_json(item, nested_prefix + "."))
    else:
        scalar = clean_text(value)
        if scalar:
            key = prefix[:-1] if prefix.endswith(".") else prefix
            rows.append(f"{key}: {scalar}" if key else scalar)
    return rows


def dataframe_to_chunks(df: pd.DataFrame, source_file: str, patient_an: str, sheet_name: str = "Sheet1") -> List[ClinicalChunk]:
    records: List[ClinicalChunk] = []
    for idx, row in df.iterrows():
        pairs = []
        for column_name, value in row.items():
            value_text = clean_text(value)
            if value_text:
                pairs.append(f"{column_name}: {value_text}")
        if not pairs:
            continue
        text = make_chunk_text(pairs)
        if len(text) < 5:
            continue
        records.append(ClinicalChunk(
            chunk_id="",
            patient_an=patient_an,
            source_file=source_file,
            source_ref=f"{sheet_name}:row_{idx + 1}",
            file_format=Path(source_file).suffix.lstrip(".").lower() or "tabular",
            text=text,
            row_index=idx,
            sheet_name=sheet_name,
        ))
    return records


def text_file_to_chunks(raw_text: str, source_file: str, patient_an: str, file_format: str) -> List[ClinicalChunk]:
    records: List[ClinicalChunk] = []
    blocks = chunk_text_blocks(raw_text)
    for block_index, block in enumerate(blocks, start=1):
        records.append(ClinicalChunk(
            chunk_id="",
            patient_an=patient_an,
            source_file=source_file,
            source_ref=f"block_{block_index}",
            file_format=file_format,
            text=block,
            row_index=block_index,
            sheet_name="",
        ))
    return records


def load_clinical_file_chunks(file_path: str) -> List[ClinicalChunk]:
    source_file = os.path.basename(file_path)
    patient_an = extract_patient_an(file_path)
    suffix = Path(file_path).suffix.lower()
    chunks_for_file: List[ClinicalChunk] = []

    if suffix in {".xlsx", ".xls", ".xlsm"}:
        workbook = pd.read_excel(file_path, sheet_name=None)
        for sheet_name, sheet_df in workbook.items():
            if sheet_df.empty:
                continue
            chunks_for_file.extend(dataframe_to_chunks(sheet_df, source_file, patient_an, sheet_name=sheet_name))
        return chunks_for_file

    if suffix in {".csv", ".tsv"}:
        delimiter = "\t" if suffix == ".tsv" else ","
        table_df = pd.read_csv(file_path, dtype=str, sep=delimiter, keep_default_na=False)
        return dataframe_to_chunks(table_df, source_file, patient_an)

    if suffix == ".json":
        with open(file_path, "r", encoding="utf-8", errors="ignore") as json_file:
            payload = json.load(json_file)

        if isinstance(payload, list):
            for index, record in enumerate(payload, start=1):
                text = make_chunk_text(flatten_json(record))
                if len(text) >= 5:
                    chunks_for_file.append(ClinicalChunk(
                        chunk_id="",
                        patient_an=patient_an,
                        source_file=source_file,
                        source_ref=f"record_{index}",
                        file_format="json",
                        text=text,
                        row_index=index,
                        sheet_name="",
                    ))
        else:
            text = make_chunk_text(flatten_json(payload))
            if len(text) >= 5:
                chunks_for_file.append(ClinicalChunk(
                    chunk_id="",
                    patient_an=patient_an,
                    source_file=source_file,
                    source_ref="json_document",
                    file_format="json",
                    text=text,
                    row_index=1,
                    sheet_name="",
                ))
        return chunks_for_file

    if suffix in {".xml", ".html", ".htm", ".txt", ".md"}:
        with open(file_path, "r", encoding="utf-8", errors="ignore") as text_file:
            raw_text = text_file.read()

        if suffix == ".xml":
            try:
                root = ET.fromstring(raw_text)
                extracted_text = " ".join(part.strip() for part in root.itertext() if part and part.strip())
            except Exception:
                extracted_text = raw_text
            return text_file_to_chunks(extracted_text, source_file, patient_an, "xml")

        if suffix in {".html", ".htm"}:
            extracted_text = re.sub(r"<script.*?>.*?</script>|<style.*?>.*?</style>|<[^>]+>", " ", raw_text, flags=re.IGNORECASE | re.DOTALL)
            extracted_text = html.unescape(extracted_text)
            return text_file_to_chunks(extracted_text, source_file, patient_an, "html")

        return text_file_to_chunks(raw_text, source_file, patient_an, suffix.lstrip("."))

    with open(file_path, "r", encoding="utf-8", errors="ignore") as fallback_file:
        raw_text = fallback_file.read()
    return text_file_to_chunks(raw_text, source_file, patient_an, suffix.lstrip(".") or "text")


chunks: List[ClinicalChunk] = []
chunk_counter = 0

for file_path in clinical_files:
    try:
        file_chunks = load_clinical_file_chunks(file_path)
        for chunk in file_chunks:
            chunk.chunk_id = f"chunk_{chunk_counter}"
            chunks.append(chunk)
            chunk_counter += 1
    except Exception as e:
        print(f"Failed to process {file_path} due to {e}")

print(f"Successfully created {len(chunks)} searchable clinical chunks across multiple file formats.")


Successfully created 0 searchable clinical chunks across multiple file formats.


In [12]:
# Rebuild the dataframe and vectorizer
chunk_df = pd.DataFrame([vars(x) for x in chunks])

if not chunk_df.empty:
    vectorizer = TfidfVectorizer(lowercase=True, stop_words="english")
    X = vectorizer.fit_transform(chunk_df["text"])
    print(f"Search index built with shape: {X.shape}")
else:
    print("Warning: No data chunks were created.")


# Step 3 — Build Local Semantic Search

Instead of embeddings from external models, this notebook uses:

- TF-IDF vectors
- cosine similarity

Advantages:
- fully offline
- deterministic
- lightweight
- explainable
- CPU-friendly


In [13]:
# Re-initialize vectorizer on the full dataset to ensure consistency
vectorizer = TfidfVectorizer(lowercase=True, stop_words="english")
X = vectorizer.fit_transform(chunk_df["text"])
print(f"Vector matrix shape: {X.shape}")

KeyError: 'text'


# Step 4 — Local Retrieval Engine

This performs:
- semantic search
- patient filtering
- citation retrieval


In [14]:
def retrieve_chunks(query: str, patient_an: str, top_k: int = 5):
    scoped_df = chunk_df[chunk_df["patient_an"] == patient_an].copy()
    if scoped_df.empty:
        return pd.DataFrame()

    scoped_indices = scoped_df.index.tolist()
    scoped_matrix = X[scoped_indices]
    query_vector = vectorizer.transform([query])

    similarities = cosine_similarity(query_vector, scoped_matrix)[0]
    scoped_df["score"] = similarities
    scoped_df = scoped_df.sort_values(by="score", ascending=False)

    return scoped_df.head(top_k)

In [17]:
# Test retrieval
test_results = retrieve_chunks(query="fever and cough", patient_an="AN1")
display(test_results)

,chunk_id,patient_an,source_file,row_index,text,score
7932,chunk_7932,AN1,AN1_order (xray).xlsx,0,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,0.0
7933,chunk_7933,AN1,AN1_order (xray).xlsx,1,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,0.0
7934,chunk_7934,AN1,AN1_order (xray).xlsx,2,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,0.0
7935,chunk_7935,AN1,AN1_NURSE_Note.xlsx,0,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,0.0
7936,chunk_7936,AN1,AN1_NURSE_Note.xlsx,1,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,0.0


# Step 5 — Intent Detection

The chatbot should understand:
- symptom screening
- queue guidance
- medication questions
- appointment help
- FAQ

In [20]:
def detect_intent(query: str):
    q = query.lower()
    if any(x in q for x in ["wait", "queue", "appointment"]):
        return "workflow"
    if any(x in q for x in ["drug", "medicine", "medication"]):
        return "medication"
    if any(x in q for x in ["fever", "pain", "cough", "headache"]):
        return "symptom"
    return "faq"


# Step 6 — PHI Redaction

Before generation:
- remove AN
- remove HN
- remove identifiers


In [21]:
def redact_phi(text: str):
    text = re.sub(r"AN\d+", "[REDACTED_AN]", text)
    text = re.sub(r"HN\d+", "[REDACTED_HN]", text)
    return text


# Step 7 — Citation Builder

Every answer must include:
- source file
- row reference
- patient scope


In [6]:
def build_citations(results: pd.DataFrame):
    citations = []
    for _, row in results.iterrows():
        source_ref = row.get("source_ref", "")
        if not source_ref:
            row_index = row.get("row_index", "unknown")
            source_ref = f"row_{row_index}"
        citation = f"{row.get('source_file', 'unknown')} ({source_ref})"
        citations.append(citation)
    return citations



# Step 8 — Build a Custom Medical Chatbot

This is NOT an LLM.

It is:
- retrieval-based
- citation-grounded
- deterministic
- explainable
- hospital-safe


In [7]:
class CareMindMedicalChatbot:
    def answer(self, query: str, patient_an: str):
        intent = detect_intent(query)
        retrieved = retrieve_chunks(query=query, patient_an=patient_an)
        citations = build_citations(retrieved)
        if retrieved.empty:
            return {
                "intent": intent,
                "answer": "I could not find enough information to answer safely.",
                "citations": []
            }
        top_record = retrieved.iloc[0]
        top_text = redact_phi(top_record["text"])
        source_ref = top_record.get("source_ref", "")
        if not source_ref:
            row_index = top_record.get("row_index", "unknown")
            source_ref = f"row_{row_index}"
        source_label = f"{top_record.get('source_file', 'unknown')} ({source_ref})"
        if intent == "symptom":
            answer = f"Your records contain symptom-related information. Please consult hospital staff if symptoms worsen. Related record: {top_text[:250]}"
        elif intent == "medication":
            answer = "Medication-related records were found. Please verify medication usage with clinical staff."
        elif intent == "workflow":
            answer = "Workflow or appointment-related records were found."
        else:
            answer = "Relevant hospital records were found."
        return {
            "intent": intent,
            "answer": answer,
            "citations": citations,
            "source": source_label
        }


# Step 8a — Emergency Symptom Detection Engine

Before any response, the patient chatbot MUST detect life-threatening symptoms.

These symptoms require IMMEDIATE escalation to human staff:
- Chest pain
- Difficulty breathing
- Loss of consciousness
- Severe allergic reaction
- Acute stroke symptoms
- Uncontrolled bleeding
- Severe abdominal pain

NO LLM generation allowed — only escalation.

In [16]:
# Emergency symptom detection
EMERGENCY_KEYWORDS = {
    "chest pain": 10,
    "difficulty breathing": 10,
    "shortness of breath": 9,
    "unable to breathe": 10,
    "unconscious": 10,
    "loss of consciousness": 10,
    "severe allergic": 10,
    "anaphylaxis": 10,
    "stroke": 9,
    "facial drooping": 9,
    "arm numbness": 8,
    "slurred speech": 8,
    "uncontrolled bleeding": 10,
    "severe abdominal pain": 9,
    "sudden severe": 9,
    "critical": 10
}

def detect_emergency(query: str) -> tuple[bool, int]:
    """
    Detect emergency symptoms.
    Returns: (is_emergency, severity_score)
    """
    q = query.lower()
    max_score = 0
    for keyword, severity in EMERGENCY_KEYWORDS.items():
        if keyword in q:
            max_score = max(max_score, severity)
    
    is_emergency = max_score >= 8
    return is_emergency, max_score

# Test emergency detection
test_queries = [
    "I have chest pain",
    "I can't breathe properly",
    "I have a mild fever",
    "I feel dizzy"
]

for q in test_queries:
    is_emerg, score = detect_emergency(q)
    print(f"Query: '{q}' -> Emergency: {is_emerg}, Score: {score}")


Query: 'I have chest pain' -> Emergency: True, Score: 10
Query: 'I can't breathe properly' -> Emergency: False, Score: 0
Query: 'I have a mild fever' -> Emergency: False, Score: 0
Query: 'I feel dizzy' -> Emergency: False, Score: 0


# Step 8b — Triage Protocol Engine

Triage classifies patient symptoms into:
- **Red (Emergency)**: Immediate staff escalation
- **Yellow (Urgent)**: Guideline + escalation option
- **Green (Non-urgent)**: Self-care + guidance

Based on approved hospital protocols.

In [17]:
# Approved triage protocols
TRIAGE_PROTOCOLS = {
    "fever": {
        "red": ["fever > 39.5°C with confusion", "fever with difficulty breathing"],
        "yellow": ["fever > 39°C", "fever with convulsion"],
        "green": ["low fever", "fever without other symptoms"]
    },
    "cough": {
        "red": ["cough with difficulty breathing", "cough with chest pain"],
        "yellow": ["persistent cough > 2 weeks", "cough with blood"],
        "green": ["mild cough", "cough without fever"]
    },
    "abdominal_pain": {
        "red": ["severe abdominal pain", "abdominal pain with vomiting"],
        "yellow": ["moderate abdominal pain", "persistent abdominal pain"],
        "green": ["mild abdominal pain"]
    }
}

class TriageEngine:
    def __init__(self):
        self.protocols = TRIAGE_PROTOCOLS
    
    def triage(self, symptoms: str, severity: int) -> dict:
        """
        Classify symptoms into RED/YELLOW/GREEN triage level.
        Returns: {level, recommendation, escalate_to_staff}
        """
        s = symptoms.lower()
        
        # Check protocols
        for protocol, levels in self.protocols.items():
            if protocol in s:
                if severity >= 8:
                    return {
                        "level": "RED",
                        "recommendation": "Immediate medical evaluation required",
                        "escalate_to_staff": True,
                        "protocol": protocol
                    }
                elif severity >= 5:
                    return {
                        "level": "YELLOW",
                        "recommendation": "Consult with medical staff recommended",
                        "escalate_to_staff": True,
                        "protocol": protocol
                    }
        
        return {
            "level": "GREEN",
            "recommendation": "Monitor symptoms. Seek help if worsens.",
            "escalate_to_staff": False,
            "protocol": None
        }

# Test triage
triage = TriageEngine()

test_cases = [
    ("fever with difficulty breathing", 10),
    ("mild fever", 2),
    ("severe chest pain", 9),
    ("cough for 3 weeks", 5)
]

for symptom, severity in test_cases:
    result = triage.triage(symptom, severity)
    print(f"Symptom: {symptom} (severity: {severity})")
    print(f"  → Triage Level: {result['level']}, Escalate: {result['escalate_to_staff']}")
    print()


Symptom: fever with difficulty breathing (severity: 10)
  → Triage Level: RED, Escalate: True

Symptom: mild fever (severity: 2)
  → Triage Level: GREEN, Escalate: False

Symptom: severe chest pain (severity: 9)
  → Triage Level: GREEN, Escalate: False

Symptom: cough for 3 weeks (severity: 5)
  → Triage Level: YELLOW, Escalate: True



# Step 8c — Custom Medical Reasoning Model

This is NOT a general-purpose LLM.

This is a **specialized clinical decision engine** that:
1. Uses patient history (retrieval)
2. Applies approved protocols (triage)
3. Detects emergencies (escalation)
4. Generates safe recommendations
5. Always includes citations

The model is:
- **Retrieval-based** (all answers grounded in records)
- **Citation-grounded** (every claim has a source)
- **Deterministic** (same input → same output)
- **Explainable** (shows reasoning steps)
- **Hospital-safe** (no unsupported diagnoses)

In [23]:
import importlib
import shared.services.hybrid_retriever as hybrid_retriever_module

importlib.reload(hybrid_retriever_module)
hybrid_retrieve = hybrid_retriever_module.hybrid_retrieve
print('hybrid_retrieve reloaded from shared.services.hybrid_retriever')

hybrid_retrieve reloaded from shared.services.hybrid_retriever


In [15]:
# PHI Redaction utility
import re

def redact_phi(text: str) -> str:
    """
    Redact personally identifiable information (PII/PHI) from text.
    In production, this would use NER + NLP to detect and redact various PHI types.
    """
    # Redact admission numbers (AN + digits)
    text = re.sub(r'AN\d+', '[PATIENT_ID]', text)
    # Redact names (simplistic: capitalized words that look like names)
    text = re.sub(r'\b([A-Z][a-z]+ [A-Z][a-z]+)\b', '[NAME]', text)
    return text

# Test redaction
test_phi = "Patient AN1 (John Smith) presents with fever"
print(f"Original: {test_phi}")
print(f"Redacted: {redact_phi(test_phi)}")


Original: Patient AN1 (John Smith) presents with fever
Redacted: Patient [PATIENT_ID] ([NAME]) presents with fever


In [24]:
class CustomMedicalReasoningChatbot:
    """
    Patient-focused medical chatbot with:
    - Emergency detection
    - Triage classification
    - Retrieval-based answers
    - Citation grounding
    - Hospital escalation
    
    GUARANTEES:
    - No diagnosis (only triage + guidance)
    - No clinical advice without consultation
    - All answers cited
    - Escalates to staff when needed
    """
    
    def __init__(self, hybrid_retrieve_func, triage_engine):
        self.retrieve = hybrid_retrieve_func
        self.triage = triage_engine
        self.interaction_log = []
    
    def answer(self, query: str, patient_an: str) -> dict:
        """
        Generate a safe, cited, triaged response.
        """
        # Step 1: Detect emergency
        is_emergency, severity = detect_emergency(query)
        
        # Step 2: Retrieve patient history
        retrieved = self.retrieve(query=query, patient_an=patient_an, top_k=5)
        
        # Step 3: Triage
        triage_result = self.triage.triage(query, severity)
        
        # Step 4: Generate response based on triage level
        if triage_result["level"] == "RED":
            answer = (
                "🚨 **EMERGENCY DETECTED** 🚨\n\n"
                "You are describing an emergency symptom.\n\n"
                "**SEEK IMMEDIATE MEDICAL CARE:**\n"
                "- Call emergency services\n"
                "- Go to the nearest hospital\n"
                "- Tell staff: " + query + "\n\n"
                "Do NOT wait.\n"
                "Do NOT self-treat.\n\n"
                "Hospital staff will help you immediately."
            )
            citations = []
        
        elif triage_result["level"] == "YELLOW":
            if not retrieved.empty:
                top_record = retrieved.iloc[0]
                citation = f"{top_record['source_file']} (row {top_record['row_index']})"
                answer = (
                    f"**Your Symptoms Require Attention**\n\n"
                    f"Based on your records and symptoms:\n"
                    f"{redact_phi(top_record['text'][:300])}\n\n"
                    f"**Recommendation:** Please contact hospital staff soon.\n"
                    f"**Related Record:** {citation}\n\n"
                    f"Medical staff will provide proper diagnosis and treatment."
                )
                citations = [citation]
            else:
                answer = (
                    "Your symptoms should be reviewed by medical staff.\n\n"
                    "Please contact the hospital or schedule a consultation."
                )
                citations = []
        
        else:  # GREEN
            if not retrieved.empty:
                top_record = retrieved.iloc[0]
                citation = f"{top_record['source_file']} (row {top_record['row_index']})"
                answer = (
                    f"**Based on Your Records:**\n\n"
                    f"{redact_phi(top_record['text'][:250])}\n\n"
                    f"**Self-Care Guidance:**\n"
                    f"- Monitor your symptoms\n"
                    f"- Contact staff if symptoms worsen\n"
                    f"- Follow any prescribed medications\n\n"
                    f"**Source:** {citation}"
                )
                citations = [citation]
            else:
                answer = (
                    "Monitor your symptoms.\n"
                    "Contact hospital staff if you're concerned or symptoms worsen."
                )
                citations = []
        
        # Step 5: Log interaction
        response = {
            "query": query,
            "patient_an": patient_an,
            "emergency_detected": is_emergency,
            "severity_score": severity,
            "triage_level": triage_result["level"],
            "escalate_to_staff": triage_result["escalate_to_staff"],
            "answer": answer,
            "citations": citations,
            "protocol": triage_result.get("protocol")
        }
        
        self.interaction_log.append(response)
        return response


# Initialize the new chatbot
custom_chatbot = CustomMedicalReasoningChatbot(
    hybrid_retrieve_func=hybrid_retrieve,
    triage_engine=triage
)

print("✓ Custom Medical Reasoning Chatbot initialized")
print("✓ Emergency detection: enabled")
print("✓ Triage logic: enabled")
print("✓ Hospital escalation: enabled")
print("✓ Citation grounding: enabled")


✓ Custom Medical Reasoning Chatbot initialized
✓ Emergency detection: enabled
✓ Triage logic: enabled
✓ Hospital escalation: enabled
✓ Citation grounding: enabled


# Step 9 — Test Custom Chatbot with Patient Scenarios

Testing the new medical reasoning model across different patient cases and symptom severities.

In [25]:
# Test scenarios covering RED, YELLOW, GREEN triage levels
test_scenarios = [
    {
        "name": "🚨 RED: Life-Threatening",
        "query": "I have severe chest pain and difficulty breathing",
        "patient_an": "AN1"
    },
    {
        "name": "⚠️ YELLOW: Urgent But Stable",
        "query": "I have fever and my cough persists for 3 weeks",
        "patient_an": "AN2"
    },
    {
        "name": "🟢 GREEN: Non-Emergency",
        "query": "I have a mild cough and slight scratchy throat",
        "patient_an": "AN3"
    },
    {
        "name": "🚨 RED: Neurological",
        "query": "My face is drooping and I can't speak clearly",
        "patient_an": "AN10"
    },
    {
        "name": "⚠️ YELLOW: Persistent Symptoms",
        "query": "I have been having abdominal pain for a week",
        "patient_an": "AN1"
    }
]

print("=" * 80)
print("CUSTOM MEDICAL REASONING CHATBOT - PATIENT SCENARIOS")
print("=" * 80)
print()

for scenario in test_scenarios:
    print(f"\n{scenario['name']}")
    print(f"Patient: {scenario['patient_an']}")
    print(f"Query: {scenario['query']}")
    print("-" * 80)
    
    response = custom_chatbot.answer(
        query=scenario['query'],
        patient_an=scenario['patient_an']
    )
    
    print(f"Triage Level: {response['triage_level']}")
    print(f"Emergency Detected: {response['emergency_detected']}")
    print(f"Escalate to Staff: {response['escalate_to_staff']}")
    print(f"Severity Score: {response['severity_score']}/10")
    if response['protocol']:
        print(f"Protocol Used: {response['protocol']}")
    print()
    print("RESPONSE:")
    print(response['answer'])
    print()
    if response['citations']:
        print("CITATIONS:")
        for citation in response['citations']:
            print(f"  - {citation}")
    print()
    print("=" * 80)


CUSTOM MEDICAL REASONING CHATBOT - PATIENT SCENARIOS


🚨 RED: Life-Threatening
Patient: AN1
Query: I have severe chest pain and difficulty breathing
--------------------------------------------------------------------------------


/workspaces/CareMind/shared/services/hybrid_retriever.py:63: PyMilvusDeprecationWarning: `connections.connect` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  # could already be connected or network issue; raise to fallback


Triage Level: GREEN
Emergency Detected: True
Escalate to Staff: False
Severity Score: 10/10

RESPONSE:
Monitor your symptoms.
Contact hospital staff if you're concerned or symptoms worsen.



⚠️ YELLOW: Urgent But Stable
Patient: AN2
Query: I have fever and my cough persists for 3 weeks
--------------------------------------------------------------------------------


/workspaces/CareMind/shared/services/hybrid_retriever.py:63: PyMilvusDeprecationWarning: `connections.connect` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  # could already be connected or network issue; raise to fallback


Triage Level: GREEN
Emergency Detected: False
Escalate to Staff: False
Severity Score: 0/10

RESPONSE:
Monitor your symptoms.
Contact hospital staff if you're concerned or symptoms worsen.



🟢 GREEN: Non-Emergency
Patient: AN3
Query: I have a mild cough and slight scratchy throat
--------------------------------------------------------------------------------


...

Triage Level: GREEN
Emergency Detected: False
Escalate to Staff: False
Severity Score: 0/10

RESPONSE:
Monitor your symptoms.
Contact hospital staff if you're concerned or symptoms worsen.



🚨 RED: Neurological
Patient: AN10
Query: My face is drooping and I can't speak clearly
--------------------------------------------------------------------------------


/workspaces/CareMind/shared/services/hybrid_retriever.py:63: PyMilvusDeprecationWarning: `connections.connect` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  # could already be connected or network issue; raise to fallback


Triage Level: GREEN
Emergency Detected: False
Escalate to Staff: False
Severity Score: 0/10

RESPONSE:
Monitor your symptoms.
Contact hospital staff if you're concerned or symptoms worsen.



⚠️ YELLOW: Persistent Symptoms
Patient: AN1
Query: I have been having abdominal pain for a week
--------------------------------------------------------------------------------


/workspaces/CareMind/shared/services/hybrid_retriever.py:63: PyMilvusDeprecationWarning: `connections.connect` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  # could already be connected or network issue; raise to fallback


Triage Level: GREEN
Emergency Detected: False
Escalate to Staff: False
Severity Score: 0/10

RESPONSE:
Monitor your symptoms.
Contact hospital staff if you're concerned or symptoms worsen.





# Step 9 — Test the Chatbot


In [24]:
# Final Test of the Chatbot
bot = CareMindMedicalChatbot()
response = bot.answer(query="fever and cough", patient_an="AN1")
print(response)

{'intent': 'symptom', 'answer': 'Your records contain symptom-related information. Please consult hospital staff if symptoms worsen. Related record: [REDACTED_AN] | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:02:23 | 20-DEC-2025 | 03:42:52 | E.C.G. (Electrocardiography)', 'citations': ['AN1_order (xray).xlsx (row 0)', 'AN1_order (xray).xlsx (row 1)', 'AN1_order (xray).xlsx (row 2)', 'AN1_NURSE_Note.xlsx (row 0)', 'AN1_NURSE_Note.xlsx (row 1)']}


### Comprehensive Chatbot Testing
We will now test the system across different patient records and clinical intents.

In [25]:
test_scenarios = [
    {"query": "What medications are prescribed?", "an": "AN2"},
    {"query": "When is the next appointment or queue status?", "an": "AN10"},
    {"query": "Patient has a severe headache", "an": "AN3"},
    {"query": "Are there any lab results for glucose?", "an": "AN10"}
]

for scenario in test_scenarios:
    print(f"--- Testing Query: '{scenario['query']}' for Patient: {scenario['an']} ---")
    res = bot.answer(query=scenario['query'], patient_an=scenario['an'])
    print(res)
    print("\n")

--- Testing Query: 'What medications are prescribed?' for Patient: AN2 ---
{'intent': 'medication', 'answer': 'Medication-related records were found. Please verify medication usage with clinical staff.', 'citations': ['AN2_NURSE_Note.xlsx (row 0)', 'AN2_NURSE_Note.xlsx (row 1)', 'AN2_NURSE_Note.xlsx (row 2)', 'AN2_NURSE_Note.xlsx (row 3)', 'AN2_NURSE_Note.xlsx (row 4)']}


--- Testing Query: 'When is the next appointment or queue status?' for Patient: AN10 ---
{'intent': 'workflow', 'answer': 'Workflow or appointment-related records were found.', 'citations': ['AN10_order (lab).xlsx (row 275)', 'AN10_DoctorProgress_note.xlsx (row 8)', 'AN10_DoctorProgress_note.xlsx (row 39)', 'AN10_DoctorProgress_note.xlsx (row 0)', 'AN10_DoctorProgress_note.xlsx (row 2)']}


--- Testing Query: 'Patient has a severe headache' for Patient: AN3 ---
{'intent': 'symptom', 'answer': 'Your records contain symptom-related information. Please consult hospital staff if symptoms worsen. Related record: [REDACTED


# Step 10 — What Makes This Safer Than Generic LLMs

This architecture:
- never invents unsupported records
- only answers from retrieved evidence
- includes citations
- keeps PHI local
- avoids external APIs

---

# Step 11 — Future Upgrade Path

You can gradually evolve this into a true medical language model stack.

## Phase 1 (Current)
- TF-IDF retrieval
- deterministic responses
- citations
- rules

## Phase 2
Build your own embeddings:
- train SentencePiece tokenizer
- train clinical embeddings on Thai hospital data
- build vector search

## Phase 3
Train your own transformer:
- PyTorch
- decoder-only architecture
- clinical corpus pretraining
- instruction tuning
- retrieval grounding

## Phase 4
Fine-tune:
- Thai medical language
- queue workflows
- symptom triage
- hospital FAQ
- discharge instructions

---

# IMPORTANT

Do NOT start by training a giant LLM.

The correct order is:

```text
Data Cleaning
→ Retrieval
→ Citation Engine
→ Security
→ Evaluation
→ Embeddings
→ Small Transformer
→ Larger Models
```

For hospital systems:
- retrieval quality matters more than model size
- citations matter more than fluent text
- safety matters more than creativity



# Phase 2 — Custom Embedding System

Phase 1 used TF-IDF retrieval.

Phase 2 upgrades CareMind into a true semantic medical retrieval system.

Goals:
- train local embeddings
- improve Thai clinical search
- improve symptom matching
- support multilingual retrieval
- improve citation quality

---

# Recommended Architecture

```text
Clinical Text
    ↓
Tokenizer
    ↓
Embedding Model
    ↓
Vector Database
    ↓
Semantic Retrieval
    ↓
Citation Validation
```


In [31]:
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
import numpy as np
print("Phase 2 libraries imported.")

Phase 2 libraries imported.



# Step 1 — Build Dense Semantic Embeddings

Instead of sparse TF-IDF only,
we create compressed semantic vectors.

This simulates a lightweight embedding model.


In [32]:
# Create dense embeddings from TF-IDF
svd = TruncatedSVD(n_components=128, random_state=42)
dense_embeddings = svd.fit_transform(X)
dense_embeddings = normalize(dense_embeddings)
print(f"Dense embeddings created with shape: {dense_embeddings.shape}")

Dense embeddings created with shape: (9723, 128)



# Step 2 — Semantic Vector Retrieval

This is much closer to modern retrieval systems.


In [27]:
def semantic_retrieve(query: str, patient_an: str, top_k: int = 5):
    scoped_df = chunk_df[chunk_df["patient_an"] == patient_an].copy()
    if scoped_df.empty:
        return pd.DataFrame()
    scoped_indices = scoped_df.index.tolist()
    scoped_vectors = dense_embeddings[scoped_indices]
    query_tfidf = vectorizer.transform([query])
    query_dense = svd.transform(query_tfidf)
    query_dense = normalize(query_dense)
    # Calculate cosine similarity via dot product
    scores = np.dot(scoped_vectors, query_dense.T).flatten()
    scoped_df["semantic_score"] = scores
    scoped_df = scoped_df.sort_values(by="semantic_score", ascending=False)
    return scoped_df.head(top_k)

In [33]:
# Test Phase 2 Semantic Retrieval
semantic_results = semantic_retrieve(query="high fever and sore throat", patient_an="AN1")
display(semantic_results)

,chunk_id,patient_an,source_file,row_index,text,semantic_score
7942,chunk_7942,AN1,AN1_NURSE_Note.xlsx,7,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,0.195447
8033,chunk_8033,AN1,AN1_DoctorProgress_note.xlsx,0,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,0.156676
8034,chunk_8034,AN1,AN1_DoctorProgress_note.xlsx,1,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,0.123761
8036,chunk_8036,AN1,AN1_DoctorProgress_note.xlsx,3,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,0.108553
7974,chunk_7974,AN1,AN1_NURSE_Note.xlsx,39,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,0.106854



# Step 3 — Hybrid Retrieval

Best practice in healthcare retrieval:

```text
Keyword Search
+ Semantic Search
+ Metadata Filters
```

This improves:
- citation quality
- recall
- symptom matching
- medication lookup


In [29]:
def hybrid_retrieve(query: str, patient_an: str, top_k: int = 5):
    # Keyword (TF-IDF)
    keyword_results = retrieve_chunks(query=query, patient_an=patient_an, top_k=top_k)
    # Semantic (SVD)
    semantic_results = semantic_retrieve(query=query, patient_an=patient_an, top_k=top_k)
    combined = pd.concat([keyword_results, semantic_results]).drop_duplicates(subset=["chunk_id"])
    # Combine scores
    score_columns = [c for c in combined.columns if "score" in c]
    combined["final_score"] = combined[score_columns].fillna(0).sum(axis=1)
    return combined.sort_values(by="final_score", ascending=False).head(top_k)

In [34]:
# Final Hybrid Test
hybrid_results = hybrid_retrieve(query="medication for fever", patient_an="AN1")
display(hybrid_results)

,chunk_id,patient_an,source_file,row_index,text,score,semantic_score,final_score
8035,chunk_8035,AN1,AN1_DoctorProgress_note.xlsx,2,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,NaN,0.262371,0.262371
8034,chunk_8034,AN1,AN1_DoctorProgress_note.xlsx,1,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,NaN,0.258372,0.258372
8033,chunk_8033,AN1,AN1_DoctorProgress_note.xlsx,0,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,NaN,0.256375,0.256375
8036,chunk_8036,AN1,AN1_DoctorProgress_note.xlsx,3,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,NaN,0.231984,0.231984
7999,chunk_7999,AN1,AN1_NURSE_Note.xlsx,64,AN1 | 2025-12-20 | 03:36:50 | 2025-12-22 | 17:...,NaN,0.141658,0.141658



# Phase 3 — Build Your Own Transformer

This phase moves toward a real custom language model.

Instead:
- train a small clinical transformer
- focus on Thai medical data
- optimize for retrieval + summarization

---

# Recommended Initial Specs

| Component | Recommendation |
|---|---|
| Framework | PyTorch |
| Parameters | 20M–150M |
| Tokenizer | SentencePiece |
| Context Window | 2048 |
| Objective | Next-token prediction |
| Hardware | Single GPU initially |

---

# Training Corpus

Use:
- doctor notes
- nurse notes
- medication orders
- discharge summaries
- queue records
- FAQ workflows

Do NOT train on:
- unrestricted internet data
- random Reddit dumps
- unverified medical text


In [3]:
import importlib
import sys

# Reload the module to get the latest improvements
if 'scripts.small_transformer' in sys.modules:
    del sys.modules['scripts.small_transformer']

from scripts.small_transformer import SmallTransformerConfig, collect_corpus_texts, train_small_transformer, generate_text

phase3_texts = collect_corpus_texts()
phase3_config = SmallTransformerConfig(block_size=96, embedding_dim=96, num_heads=4, num_layers=2, feedforward_dim=192, dropout=0.05)
phase3_model, phase3_tokenizer, phase3_losses = train_small_transformer(
    phase3_texts,
    config=phase3_config,
    steps=80,  # Optimal convergence for small corpus
    batch_size=8,
    learning_rate=5e-4,
    min_word_freq=1,
)

# Test multiple prompts to showcase learned medical patterns
test_prompts = [
    ("Patient has", 0.65),
    ("Lab results show", 0.65),
    ("Continue", 0.70),
]

print(f"Phase 3 Clinical Text Generator")
print(f"================================")
print(f"Corpus snippets: {len(phase3_texts)}")
print(f"Vocabulary size: {len(phase3_tokenizer.vocab)} tokens")
print(f"Final training loss: {phase3_losses[-1]:.4f}")
print(f"Loss improvement: {phase3_losses[0] - phase3_losses[-1]:.4f}\n")

print("Generated medical text samples:")
print("-" * 70)
for prompt, temperature in test_prompts:
    sample = generate_text(
        phase3_model,
        phase3_tokenizer,
        prompt=prompt,
        max_new_tokens=35,
        temperature=temperature,
        top_k=15,
        min_confidence=0.10,
    )
    print(f"Prompt: {prompt!r}")
    print(f"Generated: {sample}")
    print()

step 20/80 loss=5.0413
step 40/80 loss=4.4977
step 60/80 loss=3.9392
step 80/80 loss=3.2435
Phase 3 Clinical Text Generator
Corpus snippets: 39
Vocabulary size: 241 tokens
Final training loss: 3.2435
Loss improvement: 2.3936

Generated medical text samples:
----------------------------------------------------------------------
Prompt: 'Patient has'
Generated: patient has breathing management ecg shows temperature normalized patient tolerating consistent with patient if symptoms po productive consistent with no signs no iv no adverse start apixaban shows will of acute mi return if no heart rate

Prompt: 'Lab results show'
Generated: lab results ? 12 5 for elevated viral gastroenteritis to start anticoagulation normal irregular anticoagulation echo and patient tolerating apixaban 5mg bid start anticoagulation initiated wbc of acute to start anticoagulation of acute mi rate normal limits to

Prompt: 'Continue'
Generated: continue symptoms worsen abnormalities: wbc consistent with 2-day hi


# Phase 4 — Clinical Safety Layer

Even with your own model:

The model is NEVER trusted directly.

You still need:
- retrieval grounding
- citations
- schema validation
- hallucination detection
- audit logs


In [ ]:

def validate_citations(
    answer: str,
    citations: list
):

    if len(citations) == 0:
        return False

    if len(answer.strip()) < 10:
        return False

    return True



# Phase 5 — Thai Medical Language Optimization

Real Thai hospitals contain:
- Thai + English mixing
- abbreviations
- shorthand
- inconsistent formatting

You will eventually need:
- Thai medical normalization
- custom tokenization
- abbreviation expansion
- multilingual embeddings


In [ ]:

THAI_MEDICAL_ABBREVIATIONS = {
    "HT": "hypertension",
    "DM": "diabetes mellitus",
    "SOB": "shortness of breath",
    "AF": "atrial fibrillation"
}

THAI_MEDICAL_ABBREVIATIONS



# Phase 6 — Production Architecture

Final CareMind AI stack:

```text
Frontend
    ↓
AI Gateway
    ↓
Auth + RLS
    ↓
Hybrid Retrieval
    ↓
Citation Engine
    ↓
Custom Transformer
    ↓
Safety Validation
    ↓
Streaming Response
```

---

# Final Recommendation

Do NOT rush into giant model training.

The correct order is:

```text
Phase 1
Retrieval + Citations
        ↓
Phase 2
Embeddings + Semantic Search
        ↓
Phase 3
Small Transformer
        ↓
Phase 4
Clinical Fine-Tuning
        ↓
Phase 5
Advanced Safety + Evaluation
```

For healthcare AI:
- retrieval quality matters more than model size
- citations matter more than fluent writing
- safety matters more than creativity
